# bus_30min_pipeline

버스 30분 단위 일별 CSV 파일을 읽어서 월별 집계 CSV와 ZIP 파일을 생성하는 주피터 노트북입니다.

- 입력 폴더 예시: `E:/janghogeun/버스파이프라인/2022/202201/...csv`
- 월별 출력 예시: `bus_30min_monthly_2022_01.csv`
- 최종 ZIP 예시: `monthly_aggregates_0428_181500.zip`

## 1. 설정

아래 경로와 구분자(`SEP`)를 본인 파일 구조에 맞게 수정한 뒤 전체 실행하면 됩니다.

In [ ]:
from pathlib import Path

# 실행 전에 실제 폴더명에 맞게 수정하세요.
INPUT_ROOT = Path(r"E:\janghogeun\버스파이프라인")
OUTPUT_DIR = Path(r"E:\janghogeun\버스파이프라인_monthly_out")

# 원본 파일 구분자입니다. 일반 CSV면 ','로 바꾸세요.
SEP = ","

## 2. 파이프라인 함수 정의

In [ ]:
from datetime import datetime
import zipfile

import pandas as pd


REQUIRED_OUTPUT_COLUMNS = [
    "YEAR",
    "MONTH",
    "DAY",
    "HOUR",
    "HALF_HOUR",
    "STATION_ID",
    "STATION_NM",
    "GETON_CNT",
    "GETOFF_CNT",
]


# 자주 보이는 원본 컬럼명 변형을 최종 스키마명으로 통일합니다.
COLUMN_SYNONYMS = {
    "YEAR": ["YEAR", "년(YEAR)", "년 (YEAR)", "L (YEAR)"],
    "MONTH": ["MONTH", "월(MONTH)", "월 (MONTH)", "(MONTH)"],
    "DAY": ["DAY", "일(DAY)", "일 (DAY)", "(DAY)"],
    "HOUR": ["HOUR", "시간(HOUR)", "시간 (HOUR)"],
    "HALF_HOUR": ["HALF_HOUR", "분_30분단위(HALF_HOUR)", "분_30분 단위 (HALF_HOUR)"],
    "STATION_ID": ["STATION_ID", "ID(STATION_ID)", "ID (STATION_ID)"],
    "STATION_NM": ["STATION_NM", "STATIONNM", "STATION NM", "역명(STATION_NM)", "정류장명(STATION_NM)"],
    "GETON_CNT": ["GETON_CNT", "승차인원(GETON_CNT)", "승차인원(GETON CNT)", "승하차인원(GETON_CNT)"],
    "GETOFF_CNT": ["GETOFF_CNT", "하차인원(GETOFF_CNT)", "하차인원(GETOFF CNT)"],
}


def normalize_col_name(col: str) -> str:
    return str(col).strip().replace("\ufeff", "")


def harmonize_columns(df: pd.DataFrame) -> pd.DataFrame:
    src_cols = {normalize_col_name(c): c for c in df.columns}
    rename_map = {}

    for target, candidates in COLUMN_SYNONYMS.items():
        found = None
        for candidate in candidates:
            candidate_norm = normalize_col_name(candidate)
            if candidate_norm in src_cols:
                found = src_cols[candidate_norm]
                break

        if found is not None:
            rename_map[found] = target

    df = df.rename(columns=rename_map)
    missing = [c for c in REQUIRED_OUTPUT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return df[REQUIRED_OUTPUT_COLUMNS].copy()


def read_daily_csv(file_path: Path, sep: str = ",") -> pd.DataFrame:
    last_error = None

    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(file_path, engine="python", dtype=str, encoding=encoding, sep=sep)
            break
        except UnicodeDecodeError as e:
            last_error = e
    else:
        raise last_error

    df = harmonize_columns(df)

    for col in ["GETON_CNT", "GETOFF_CNT"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("'", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


def aggregate_monthly(month_df: pd.DataFrame) -> pd.DataFrame:
    # 월별 집계는 정류장 + 30분 시간대 단위로 수행하므로 DAY는 제거합니다.
    group_cols = ["YEAR", "MONTH", "HOUR", "HALF_HOUR", "STATION_ID", "STATION_NM"]

    return (
        month_df.groupby(group_cols, dropna=False, as_index=False)[["GETON_CNT", "GETOFF_CNT"]]
        .sum()
        .sort_values(group_cols)
    )


def zip_output_folder(output_dir: Path) -> Path:
    ts = datetime.now().strftime("%m%d_%H%M%S")
    zip_path = output_dir.parent / f"monthly_aggregates_{ts}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in output_dir.glob("*.csv"):
            zf.write(file, arcname=file.name)

    return zip_path


def discover_month_folders(input_root: Path) -> list[Path]:
    # Case A: input_root/202201, input_root/202202, ...
    direct_months = [
        p for p in input_root.iterdir()
        if p.is_dir() and len(p.name) == 6 and p.name.isdigit()
    ]
    if direct_months:
        return sorted(direct_months)

    # Case B: input_root/2022/202201, input_root/2022/202202, ...
    year_dirs = [
        p for p in input_root.iterdir()
        if p.is_dir() and len(p.name) == 4 and p.name.isdigit()
    ]
    nested_months = []
    for year_dir in year_dirs:
        nested_months.extend(
            p for p in year_dir.iterdir()
            if p.is_dir() and len(p.name) == 6 and p.name.isdigit()
        )

    return sorted(nested_months)


def run_pipeline(input_root: Path, output_dir: Path, sep: str = ",") -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    month_dirs = discover_month_folders(input_root)
    if not month_dirs:
        raise FileNotFoundError(
            f"No month folders found under: {input_root}. "
            "Expected folders like 202201, 202202, ..."
        )

    written_files = []

    for month_dir in month_dirs:
        # 아래 두 구조를 모두 처리합니다.
        # 1) input_root/2022/202201/TBD...
        # 2) input_root/2022/202201/202201/TBD...
        nested_same_month = month_dir / month_dir.name
        data_dir = nested_same_month if nested_same_month.is_dir() else month_dir

        daily_files = sorted(data_dir.glob("TBD_TRANSIT_STAT_BUS_*.csv"))
        if not daily_files:
            daily_files = sorted(data_dir.glob("*.csv"))

        if not daily_files:
            print(f"[WARN] No daily csv found in {data_dir}")
            continue

        monthly_frames = []
        for file in daily_files:
            try:
                monthly_frames.append(read_daily_csv(file, sep=sep))
            except Exception as e:
                print(f"[WARN] Skip file due to parse/schema issue: {file} -> {e}")

        if not monthly_frames:
            print(f"[WARN] No valid daily file in {data_dir}")
            continue

        month_df = pd.concat(monthly_frames, ignore_index=True)
        monthly_agg = aggregate_monthly(month_df)

        yy = month_dir.name[:4]
        mm = month_dir.name[4:6]
        out_file = output_dir / f"bus_30min_monthly_{yy}_{mm}.csv"
        monthly_agg.to_csv(out_file, index=False, encoding="utf-8-sig")
        written_files.append(out_file)
        print(f"[OK] {out_file} rows={len(monthly_agg):,}")

    if not written_files:
        raise RuntimeError("No monthly output was generated. Check folder structure, columns, or separator.")

    zip_path = zip_output_folder(output_dir)
    print("\n=== Pipeline Completed ===")
    print(f"Monthly files: {len(written_files)}")
    print(f"Zip: {zip_path}")

## 3. 실행

In [ ]:
run_pipeline(input_root=INPUT_ROOT, output_dir=OUTPUT_DIR, sep=SEP)